<a href="https://colab.research.google.com/github/SaiSiri05/MachineLearning_Labs/blob/main/Using_KNN_for_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
from sklearn import metrics, neighbors
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.tokenize import word_tokenize
import re
import string

# Download NLTK data if not already downloaded
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

def text_preprocessing(text, remove_stopwords=True, lemmatize=True, stemmer=False):
    # Ensure text is a string
    if not isinstance(text, str):
        return ""

    # Lowercase the text
    text = text.lower()
    # Remove punctuation
    text = ''.join([char for char in text if char not in string.punctuation])
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Tokenize
    tokens = word_tokenize(text)

    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
    elif stemmer:
        stemmer_obj = PorterStemmer()
        tokens = [stemmer_obj.stem(word) for word in tokens]

    return ' '.join(tokens)

def createBagOfWords(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False):
    # Apply preprocessing to training and testing messages
    X_train_processed = X_train.apply(lambda text: text_preprocessing(text, remove_stopwords, lemmatize, stemmer))
    X_test_processed = X_test.apply(lambda text: text_preprocessing(text, remove_stopwords, lemmatize, stemmer))

    vectorizer = CountVectorizer()
    X_train_bow = vectorizer.fit_transform(X_train_processed)
    X_test_bow = vectorizer.transform(X_test_processed)
    return X_train_bow, X_test_bow

def createTFIDF(X_train, X_test, remove_stopwords=True, lemmatize=True, stemmer=False):
    # Apply preprocessing to training and testing messages
    X_train_processed = X_train.apply(lambda text: text_preprocessing(text, remove_stopwords, lemmatize, stemmer))
    X_test_processed = X_test.apply(lambda text: text_preprocessing(text, remove_stopwords, lemmatize, stemmer))

    vectorizer = TfidfVectorizer()
    X_train_tfidf = vectorizer.fit_transform(X_train_processed)
    X_test_tfidf = vectorizer.transform(X_test_processed)
    return X_train_tfidf, X_test_tfidf

def bow_knn():

    training_data = pd.read_csv('spam.csv', encoding='latin-1')

    if training_data.shape[1] > 2:
        training_data = training_data.iloc[:, :2]

    training_data.columns = ['Category', 'Message']
    training_data = training_data.dropna()

    training_data['Category'] = training_data['Category'].map({'ham': 0, 'spam': 1})

    X_train, X_test, y_train, y_test = train_test_split(
        training_data["Message"],
        training_data["Category"],
        test_size=0.2,
        random_state=42
    )

    X_train, X_test = createBagOfWords(
        X_train,
        X_test,
        remove_stopwords=True,
        lemmatize=True,
        stemmer=False
    )

    params = [
        (3, 'uniform', 'euclidean'),
        (5, 'distance', 'manhattan'),
        (7, 'uniform', 'cosine')
    ]

    for k, weight, metric in params:

        knn = neighbors.KNeighborsClassifier(
            n_neighbors=k,
            weights=weight,
            metric=metric
        )

        knn.fit(X_train, y_train)

        predicted = knn.predict(X_test)

        acc = metrics.accuracy_score(y_test, predicted)

        print(f'K={k}, Weights={weight}, Metric={metric}')
        print('Accuracy = ' + str(acc * 100) + '%')

        scores = cross_val_score(knn, X_train, y_train, cv=3)

        print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
        print(scores)
        print('\n')

    return predicted, y_test

def tfidf_knn():

    training_data = pd.read_csv('spam.csv', encoding='latin-1')

    if training_data.shape[1] > 2:
        training_data = training_data.iloc[:, :2]

    training_data.columns = ['Category', 'Message']
    training_data = training_data.dropna()

    training_data['Category'] = training_data['Category'].map({'ham': 0, 'spam': 1})

    X_train, X_test, y_train, y_test = train_test_split(
        training_data["Message"],
        training_data["Category"],
        test_size=0.2,
        random_state=42
    )

    X_train, X_test = createTFIDF(
        X_train,
        X_test,
        remove_stopwords=True,
        lemmatize=True,
        stemmer=False
    )

    params = [
        (3, 'uniform', 'cosine'),
        (5, 'distance', 'euclidean'),
        (9, 'distance', 'manhattan')
    ]

    for k, weight, metric in params:

        knn = neighbors.KNeighborsClassifier(
            n_neighbors=k,
            weights=weight,
            metric=metric
        )

        knn.fit(X_train, y_train)

        predicted = knn.predict(X_test)

        acc = metrics.accuracy_score(y_test, predicted)

        print(f'K={k}, Weights={weight}, Metric={metric}')
        print('Accuracy = ' + str(acc * 100) + '%')

        scores = cross_val_score(knn, X_train, y_train, cv=3)

        print("Cross Validation Accuracy: %0.2f" % (scores.mean()))
        print(scores)
        print('\n')

    return predicted, y_test

bow_knn()
tfidf_knn()

K=3, Weights=uniform, Metric=euclidean
Accuracy = 93.90134529147983%
Cross Validation Accuracy: 0.92
[0.92261104 0.92059219 0.91919192]


K=5, Weights=distance, Metric=manhattan
Accuracy = 94.79820627802691%
Cross Validation Accuracy: 0.92
[0.91991925 0.92799462 0.92659933]


K=7, Weights=uniform, Metric=cosine
Accuracy = 96.95067264573991%
Cross Validation Accuracy: 0.96
[0.96096904 0.95289367 0.96094276]


K=3, Weights=uniform, Metric=cosine
Accuracy = 97.66816143497758%
Cross Validation Accuracy: 0.96
[0.96702557 0.9602961  0.96363636]


K=5, Weights=distance, Metric=euclidean
Accuracy = 94.52914798206278%
Cross Validation Accuracy: 0.92
[0.91453567 0.92059219 0.91851852]


K=9, Weights=distance, Metric=manhattan
Accuracy = 86.90582959641256%
Cross Validation Accuracy: 0.91
[0.9051144  0.90444145 0.90841751]




(array([0, 0, 0, ..., 0, 0, 0]),
 3245    0
 944     0
 1044    0
 2484    0
 812     0
        ..
 4264    0
 2439    0
 5556    0
 4205    0
 4293    0
 Name: Category, Length: 1115, dtype: int64)